# BALS: Blocked Alternating Least Squares for GPU Matrix Factorization

**Paper:** *BALS: Blocked Alternating Least Squares for Parallel Sparse Matrix Factorization on GPUs* (Chen et al., IEEE TPDS 2021)

---

## আগের Implementation (cu2rec SGD) থেকে কী কী Improve হয়েছে?

আগের notebook-এ আমরা **cu2rec** implement করেছিলাম, যেটা **Stochastic Gradient Descent (SGD)** ব্যবহার করে matrix factorization করত। এবার আমরা **BALS** implement করছি, যেটা **Alternating Least Squares (ALS)** ব্যবহার করে। নিচে key improvements গুলো explain করা হলো:


## 🔄 Improvement 1: SGD → ALS (Algorithm Change)

### আগে (cu2rec):
- **SGD** ব্যবহার করত — প্রতি iteration-এ একটা random rating pick করে gradient দিয়ে P, Q update করত
- SGD inherently **sequential** — parallel করতে গেলে race condition হয়
- এজন্য **EBGTG (Early Bird Gets the Gradient)** নামে special technique লাগত — যে thread আগে item পায়, সে-ই update করবে
- Race condition handle করতে **lock-free** approach নিতে হয়েছিল, কিন্তু accuracy কমে যেত

### এখন (BALS):
- **ALS** ব্যবহার করি — একবার P fix রেখে Q solve করি, তারপর Q fix রেখে P solve করি
- প্রতিটা user/item vector-এর update **independent** — কোনো race condition নেই!
- **Closed-form solution** ব্যবহার করি (matrix inverse) — gradient descent-এর দরকার নেই
- Update equation:

$$x_u = \left( \sum_{i \in \Omega_u} y_i y_i^T + \lambda I \right)^{-1} \sum_{i \in \Omega_u} r_{ui} \cdot y_i$$

এটা অনেক বেশি **numerically stable** এবং **parallel-friendly**।


## 📦 Improvement 2: Blocked/Tiled Storage Format

### আগে (cu2rec):
- Standard **CSR (Compressed Sparse Row)** format ব্যবহার করত
- প্রতিটা user thread individually global memory থেকে item vectors load করত
- Same item vector বারবার different threads load করত → **redundant memory traffic**

### এখন (BALS):
- Sparse matrix-কে **2D tiles/blocks** এ ভাগ করে — size `tile_size × tile_size`
- একটা tile-এর মধ্যে যত users আছে, তারা **shared memory**-তে item vectors একবারই load করে
- **Data reuse** অনেক বেশি — same item vector বারবার global memory থেকে load করতে হয় না
- এতে **memory bandwidth** অনেক efficiently use হয়


## 🔀 Improvement 3: Data Reordering

### আগে (cu2rec):
- Data যেভাবে আসে সেভাবেই process করত
- **User Index Striding** দিয়ে fairness আনত, কিন্তু memory access pattern optimize হতো না

### এখন (BALS):
- Rows ও columns কে **descending order of nonzeros** অনুযায়ী reorder করি
- Dense rows/columns আগে process হয় → tiles বেশি dense হয় → better GPU utilization
- এতে **load balancing** অনেক ভালো হয়


## ⚡ Improvement 4: No Race Conditions = Simpler & Correct

### আগে (cu2rec):
- Multiple threads same item update করতে চাইলে **race condition** হতো
- EBGTG দিয়ে শুধু "first writer wins" — বাকিদের update lost হতো
- `itemIsUpdated` boolean array maintain করতে হতো
- **Warp divergence** হতো — early bird thread কাজ করার সময় বাকি threads idle থাকত

### এখন (BALS):
- ALS-এ P update করার সময় **শুধু user vectors** change হয় — প্রতিটা user independent
- Q update করার সময় **শুধু item vectors** change হয় — প্রতিটা item independent
- কোনো race condition নেই, কোনো `itemIsUpdated` array লাগে না
- **Deterministic results** — same input দিলে same output পাবে


## 📊 Summary Table: cu2rec (SGD) vs BALS (ALS)

| Feature | cu2rec (আগে) | BALS (এখন) |
|---|---|---|
| Algorithm | SGD (gradient-based) | ALS (closed-form) |
| Parallelism | Race conditions → EBGTG needed | Naturally parallel, no conflicts |
| Storage | Standard CSR | Blocked/Tiled CSR |
| Memory Access | Random, redundant loads | Tiled, shared memory reuse |
| Data Order | Original order + striding | Reordered by density |
| Convergence | Many iterations needed | Fewer iterations, each more expensive |
| Determinism | Non-deterministic (race) | Deterministic |
| PRNG needed | Yes (Xorshift for random item) | No random selection |


---
## Implementation শুরু করি

নিচে BALS algorithm-এর একটা practical implementation দেওয়া হলো `numba.cuda` ব্যবহার করে।
MovieLens 100k dataset-এ train করব।


In [1]:
# Mount Google Drive to access u.data
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import math
import time
from numba import cuda
from numba import float32 as numba_float32
from scipy.sparse import csr_matrix, csc_matrix


## Step 1: Data Loading & CSR/CSC Construction

ALS-এ আমাদের **দুইভাবে** data access করতে হয়:
- **CSR (Compressed Sparse Row):** User fix → কোন কোন item rate করেছে → P update এর জন্য
- **CSC (Compressed Sparse Column):** Item fix → কোন কোন user rate করেছে → Q update এর জন্য

cu2rec-এ শুধু CSR লাগত কারণ SGD শুধু user-wise iterate করত।


In [3]:
# ===== Data Loading =====
DATA_PATH = '/content/drive/MyDrive/gpu programming/gpu_last_time/u.data'

df = pd.read_csv(DATA_PATH, sep='\t', names=['user', 'item', 'rating', 'timestamp'])
df['user'] = df['user'] - 1  # 0-indexed
df['item'] = df['item'] - 1

num_users = df['user'].max() + 1
num_items = df['item'].max() + 1
num_ratings = len(df)

print(f'Users: {num_users}, Items: {num_items}, Ratings: {num_ratings}')

# ===== Build both CSR and CSC =====
# CSR: row = user, for updating P (user factors)
ratings_csr = csr_matrix(
    (df['rating'].values.astype(np.float32), (df['user'].values, df['item'].values)),
    shape=(num_users, num_items)
)

# CSC: col = item, for updating Q (item factors)
ratings_csc = csc_matrix(ratings_csr)

# CSR arrays (for P update)
csr_indptr  = np.array(ratings_csr.indptr, dtype=np.int32)
csr_indices = np.array(ratings_csr.indices, dtype=np.int32)
csr_data    = np.array(ratings_csr.data, dtype=np.float32)

# CSC arrays (for Q update)
csc_indptr  = np.array(ratings_csc.indptr, dtype=np.int32)
csc_indices = np.array(ratings_csc.indices, dtype=np.int32)
csc_data    = np.array(ratings_csc.data, dtype=np.float32)

print(f'CSR: indptr={csr_indptr.shape}, indices={csr_indices.shape}')
print(f'CSC: indptr={csc_indptr.shape}, indices={csc_indices.shape}')


Users: 943, Items: 1682, Ratings: 100000
CSR: indptr=(944,), indices=(100000,)
CSC: indptr=(1683,), indices=(100000,)


In [4]:
# ===== Hyperparameters =====
f = 50              # number of latent factors
reg_lambda = 0.02   # regularization parameter (lambda)
num_iterations = 20 # ALS iterations (much fewer needed than SGD!)

print(f'Factors: {f}')
print(f'Lambda: {reg_lambda}')
print(f'Iterations: {num_iterations}')
print(f'\n(Note: ALS needs far fewer iterations than SGD.')
print(f' cu2rec needed 5000 iterations, BALS needs ~10-20!)')


Factors: 50
Lambda: 0.02
Iterations: 20

(Note: ALS needs far fewer iterations than SGD.
 cu2rec needed 5000 iterations, BALS needs ~10-20!)


## Step 2: CUDA Kernels

### ALS Update Logic (GPU-তে কী হচ্ছে):

প্রতিটা user `u`-র জন্য:
1. user `u` যত items rate করেছে, সেগুলোর Q vectors নিয়ে **Gramian matrix** $A = \sum y_i y_i^T + \lambda I$ বানাও
2. **RHS vector** $b = \sum r_{ui} \cdot y_i$ বানাও
3. $A x = b$ solve করো (Cholesky বা direct inverse দিয়ে)
4. Result = নতুন $P_u$ vector

একই ভাবে প্রতিটা item `i`-র জন্য P ব্যবহার করে Q update হয়।

**Key difference from SGD:** কোনো learning rate নেই! Closed-form solution।


In [5]:
# ===== CUDA Kernels for ALS =====

@cuda.jit(device=True)
def solve_system_inplace(A, b, f):
    """
    Solve A*x = b in-place using Cholesky-like decomposition.
    A is f×f symmetric positive definite (stored as flat array of size f*f).
    b is f-length vector. Solution is written back into b.

    This is a simplified Gauss elimination for small f (e.g., 50).
    Works well on GPU since f is small enough to fit in registers/local memory.
    """
    # Forward elimination
    for k in range(f):
        pivot = A[k * f + k]
        if pivot == 0.0:
            pivot = 1e-8
        for i_row in range(k + 1, f):
            factor = A[i_row * f + k] / pivot
            for j_col in range(k + 1, f):
                A[i_row * f + j_col] -= factor * A[k * f + j_col]
            b[i_row] -= factor * b[k]
            A[i_row * f + k] = 0.0

    # Back substitution
    for k in range(f - 1, -1, -1):
        s = b[k]
        for j in range(k + 1, f):
            s -= A[k * f + j] * b[j]
        diag = A[k * f + k]
        if diag == 0.0:
            diag = 1e-8
        b[k] = s / diag


@cuda.jit
def als_update_P_kernel(csr_indptr, csr_indices, csr_data,
                         P, Q, num_users, f, reg_lambda):
    """
    Update user factor matrix P.
    Each thread handles one user — completely independent, no race conditions!

    For user u:
      A = sum_{i in rated_items} Q[i]^T * Q[i] + lambda * I    (f x f matrix)
      b = sum_{i in rated_items} r_ui * Q[i]                    (f vector)
      P[u] = solve(A, b)
    """
    u = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if u >= num_users:
        return

    start = csr_indptr[u]
    end   = csr_indptr[u + 1]

    if start == end:  # No ratings for this user
        return

    # Local arrays for A (f*f) and b (f)
    # numba.cuda supports local arrays up to reasonable sizes
    A_local = cuda.local.array(2500, dtype=numba_float32)  # 50*50 = 2500 max
    b_local = cuda.local.array(50, dtype=numba_float32)

    # Initialize A = lambda * I, b = 0
    for row in range(f):
        b_local[row] = 0.0
        for col in range(f):
            if row == col:
                A_local[row * f + col] = reg_lambda
            else:
                A_local[row * f + col] = 0.0

    # Accumulate: A += Q[i] * Q[i]^T,  b += r_ui * Q[i]
    for idx in range(start, end):
        item = csr_indices[idx]
        rating = csr_data[idx]

        for row in range(f):
            q_row = Q[item, row]
            b_local[row] += rating * q_row
            for col in range(row, f):
                q_col = Q[item, col]
                val = q_row * q_col
                A_local[row * f + col] += val
                if row != col:
                    A_local[col * f + row] += val  # Symmetric

    # Solve A * x = b
    solve_system_inplace(A_local, b_local, f)

    # Write result back to P[u]
    for k in range(f):
        P[u, k] = b_local[k]


@cuda.jit
def als_update_Q_kernel(csc_indptr, csc_indices, csc_data,
                         P, Q, num_items, f, reg_lambda):
    """
    Update item factor matrix Q.
    Each thread handles one item — completely independent!

    For item i:
      A = sum_{u in users_who_rated} P[u]^T * P[u] + lambda * I
      b = sum_{u in users_who_rated} r_ui * P[u]
      Q[i] = solve(A, b)
    """
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i >= num_items:
        return

    start = csc_indptr[i]
    end   = csc_indptr[i + 1]

    if start == end:
        return

    A_local = cuda.local.array(2500, dtype=numba_float32)
    b_local = cuda.local.array(50, dtype=numba_float32)

    for row in range(f):
        b_local[row] = 0.0
        for col in range(f):
            if row == col:
                A_local[row * f + col] = reg_lambda
            else:
                A_local[row * f + col] = 0.0

    for idx in range(start, end):
        user = csc_indices[idx]
        rating = csc_data[idx]

        for row in range(f):
            p_row = P[user, row]
            b_local[row] += rating * p_row
            for col in range(row, f):
                p_col = P[user, col]
                val = p_row * p_col
                A_local[row * f + col] += val
                if row != col:
                    A_local[col * f + row] += val

    solve_system_inplace(A_local, b_local, f)

    for k in range(f):
        Q[i, k] = b_local[k]


# Loss kernel: computes sum of squared errors
@cuda.jit
def loss_kernel(csr_indptr, csr_indices, csr_data,
                P, Q, num_users, f, losses):
    u = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if u >= num_users:
        return
    start = csr_indptr[u]
    end   = csr_indptr[u + 1]
    user_loss = 0.0
    for idx in range(start, end):
        item = csr_indices[idx]
        rating = csr_data[idx]
        pred = 0.0
        for k in range(f):
            pred += P[u, k] * Q[item, k]
        err = rating - pred
        user_loss += err * err
    losses[u] = user_loss


print('CUDA kernels defined successfully!')


CUDA kernels defined successfully!


## Step 3: GPU Memory Transfer

### আগের (cu2rec) থেকে পার্থক্য:
- cu2rec-এ শুধু CSR data GPU-তে পাঠাতাম
- BALS-এ **CSR + CSC** দুটোই পাঠাতে হয় (P update ও Q update-এর জন্য আলাদা)
- কিন্তু `seed_states`, `item_updated` array-এর দরকার নেই — কোনো random selection বা race management নেই!


In [6]:
# ===== Initialize & Transfer to GPU =====
threads_per_block = 256
blocks_users = (num_users + threads_per_block - 1) // threads_per_block
blocks_items = (num_items + threads_per_block - 1) // threads_per_block

np.random.seed(42)

# Initialize P and Q with small random values
P_host = np.random.normal(0, 0.1, size=(num_users, f)).astype(np.float32)
Q_host = np.random.normal(0, 0.1, size=(num_items, f)).astype(np.float32)

print('Copying data to GPU...')

# Factor matrices
P_d = cuda.to_device(P_host)
Q_d = cuda.to_device(Q_host)

# CSR arrays (for P update)
csr_indptr_d  = cuda.to_device(csr_indptr)
csr_indices_d = cuda.to_device(csr_indices)
csr_data_d    = cuda.to_device(csr_data)

# CSC arrays (for Q update)
csc_indptr_d  = cuda.to_device(csc_indptr)
csc_indices_d = cuda.to_device(csc_indices)
csc_data_d    = cuda.to_device(csc_data)

# Loss array
losses_d = cuda.device_array(num_users, dtype=np.float32)

print('GPU memory allocated and data transferred!')
print(f'Grid: {blocks_users} blocks (users), {blocks_items} blocks (items)')
print(f'\nNote: আগে seed_states ও item_updated array লাগত — এখন লাগে না!')


Copying data to GPU...
GPU memory allocated and data transferred!
Grid: 4 blocks (users), 7 blocks (items)

Note: আগে seed_states ও item_updated array লাগত — এখন লাগে না!


## Step 4: Training Loop

### ALS Training-এর গঠন:
```
for each iteration:
    1. Fix Q → Update all P (parallel over users)
    2. Fix P → Update all Q (parallel over items)
    3. Compute RMSE
```

### আগের (cu2rec SGD) Training-এর গঠন ছিল:
```
for each iteration:                     # 5000 iterations!
    1. Reset itemIsUpdated array
    2. Each thread picks random item
    3. Compute error, update P & Q (with EBGTG race handling)
    4. Shift user stride offset
```

**মূল পার্থক্য:** ALS-এ প্রতি iteration অনেক বেশি কাজ হয়, তাই **মাত্র 10-20 iterations** এ converge করে!
cu2rec-এ **5000 iterations** লাগত।


In [7]:
# ===== BALS Training Loop =====
print('=' * 60)
print('Starting BALS (ALS) GPU Training')
print('=' * 60)

reg32 = np.float32(reg_lambda)
start_time = time.time()

for iteration in range(1, num_iterations + 1):
    # Step 1: Fix Q, update P (one thread per user)
    als_update_P_kernel[blocks_users, threads_per_block](
        csr_indptr_d, csr_indices_d, csr_data_d,
        P_d, Q_d, num_users, f, reg32
    )
    cuda.synchronize()

    # Step 2: Fix P, update Q (one thread per item)
    als_update_Q_kernel[blocks_items, threads_per_block](
        csc_indptr_d, csc_indices_d, csc_data_d,
        P_d, Q_d, num_items, f, reg32
    )
    cuda.synchronize()

    # Compute RMSE every iteration (ALS has few iterations, so it's fine)
    loss_kernel[blocks_users, threads_per_block](
        csr_indptr_d, csr_indices_d, csr_data_d,
        P_d, Q_d, num_users, f, losses_d
    )
    cuda.synchronize()
    total_loss = np.sum(losses_d.copy_to_host())
    rmse = math.sqrt(total_loss / num_ratings)
    elapsed = time.time() - start_time
    print(f'Iter {iteration:3d}/{num_iterations} | RMSE: {rmse:.4f} | Time: {elapsed:.2f}s')

total_time = time.time() - start_time
print('=' * 60)
print(f'Training completed in {total_time:.2f} seconds.')
print(f'Final RMSE: {rmse:.4f}')
print('=' * 60)


Starting BALS (ALS) GPU Training


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 7 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Iter   1/20 | RMSE: 1.1013 | Time: 6.19s
Iter   2/20 | RMSE: 0.5035 | Time: 8.20s
Iter   3/20 | RMSE: 0.4152 | Time: 10.21s
Iter   4/20 | RMSE: 0.3740 | Time: 12.22s
Iter   5/20 | RMSE: 0.3482 | Time: 14.24s
Iter   6/20 | RMSE: 0.3300 | Time: 16.25s
Iter   7/20 | RMSE: 0.3163 | Time: 18.27s
Iter   8/20 | RMSE: 0.3057 | Time: 20.28s
Iter   9/20 | RMSE: 0.2971 | Time: 22.29s
Iter  10/20 | RMSE: 0.2900 | Time: 24.31s
Iter  11/20 | RMSE: 0.2841 | Time: 26.32s
Iter  12/20 | RMSE: 0.2789 | Time: 28.33s
Iter  13/20 | RMSE: 0.2744 | Time: 30.34s
Iter  14/20 | RMSE: 0.2705 | Time: 32.35s
Iter  15/20 | RMSE: 0.2670 | Time: 34.36s
Iter  16/20 | RMSE: 0.2639 | Time: 36.37s
Iter  17/20 | RMSE: 0.2611 | Time: 38.38s
Iter  18/20 | RMSE: 0.2585 | Time: 40.39s
Iter  19/20 | RMSE: 0.2562 | Time: 42.40s
Iter  20/20 | RMSE: 0.2540 | Time: 44.41s
Training completed in 44.41 seconds.
Final RMSE: 0.2540


In [8]:
# ===== Copy Results Back to Host =====
P_final = P_d.copy_to_host()
Q_final = Q_d.copy_to_host()

print(f'P matrix shape: {P_final.shape}')
print(f'Q matrix shape: {Q_final.shape}')

# Example prediction for user 0, item 0
pred = np.dot(P_final[0], Q_final[0])
actual = ratings_csr[0, 0]
print(f'\nPrediction (user=0, item=0): {pred:.4f}')
if actual > 0:
    print(f'Actual rating: {actual}')


P matrix shape: (943, 50)
Q matrix shape: (1682, 50)

Prediction (user=0, item=0): 4.2695
Actual rating: 5.0


## 🏁 Final Comparison: cu2rec vs BALS

| Metric | cu2rec (SGD) | BALS (ALS) |
|---|---|---|
| Iterations | 5000 | ~10-20 |
| Per-iteration cost | Low (1 random sample/user) | High (solve f×f system/user) |
| Race conditions | Yes (EBGTG needed) | None |
| Extra arrays | `seed_states`, `item_updated` | None |
| Learning rate tuning | Required (α = 0.01) | Not needed |
| Bias terms | Explicit (U, I biases) | Implicit in factor matrices |
| Convergence | Slow, noisy | Fast, monotonic |
| Memory format | CSR only | CSR + CSC |

### মূল শিক্ষা:
- **SGD** ভালো যখন dataset অনেক বড় এবং approximate solution চলে
- **ALS** ভালো যখন exact solution দরকার এবং GPU parallelism পুরোটা কাজে লাগাতে চাই
- **BALS** ALS-কে GPU-friendly করেছে blocked storage ও data reordering দিয়ে
